In [2]:
import os
import glob

import numpy as np
import pandas as pd
import kagglehub

In [7]:
from datasets import load_dataset

medic = load_dataset(
    "QCRI/MEDIC",
    split="train",
    streaming=True
)

print(medic.features)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


{'image_id': Value('string'), 'event_name': Value('string'), 'image_path': Value('string'), 'damage_severity': ClassLabel(names=['little_or_none', 'mild', 'severe']), 'informative': ClassLabel(names=['informative', 'not_informative']), 'humanitarian': ClassLabel(names=['affected_injured_or_dead_people', 'infrastructure_and_utility_damage', 'not_humanitarian', 'rescue_volunteering_or_donation_effort']), 'disaster_types': ClassLabel(names=['earthquake', 'flood', 'hurricane', 'fire', 'landslide', 'not_disaster', 'other_disaster']), 'image': Image(mode=None, decode=True)}


In [5]:
crisismmd_df = pd.concat(
    [
        pd.read_csv(file, sep="\t")
        for file in annotation_files
    ],
    ignore_index=True
)

print("CrisisMMD rows:", len(crisismmd_df))
print("CrisisMMD columns:")
print(crisismmd_df.columns.tolist())

CrisisMMD rows: 18126
CrisisMMD columns:
['tweet_id', 'image_id', 'text_info', 'text_info_conf', 'image_info', 'image_info_conf', 'text_human', 'text_human_conf', 'image_human', 'image_human_conf', 'image_damage', 'image_damage_conf', 'tweet_text', 'image_url', 'image_path']


In [6]:
crisismmd_text_lookup = (
    crisismmd_df[
        ["image_id", "tweet_text"]
    ]
    .drop_duplicates("image_id")
    .set_index("image_id")["tweet_text"]
    .to_dict()
)

print("Unique CrisisMMD image IDs:", len(crisismmd_text_lookup))

Unique CrisisMMD image IDs: 18102


In [8]:
matched_rows = []

for item in medic:

    filename = os.path.basename(item["image_path"])
    medic_image_id = os.path.splitext(filename)[0]

    if medic_image_id in crisismmd_text_lookup:

        matched_rows.append({
            "image_id": medic_image_id,
            "tweet_text": crisismmd_text_lookup[medic_image_id],
            "image_path": item["image_path"],
            "damage_severity": item["damage_severity"],
            "disaster_types": item["disaster_types"]
        })

print("Matched samples:", len(matched_rows))

Matched samples: 11623


In [10]:
matched_rows.head

AttributeError: 'list' object has no attribute 'head'